In [39]:
import pandas as pd
from pathlib import Path

# 获取项目根目录
project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

# 使用之前已经定义的项目根目录
file_path = project_root / "data/processed/model_dataset.csv"

# 确认文件存在
print("文件是否存在：", file_path.exists())

# 读取建模数据
df = pd.read_csv(file_path)

print("数据规模：", df.shape)
print("订单ID是否唯一：", df["order_id"].is_unique)
print("标签分布：")
print(df["is_delayed"].value_counts())

文件是否存在： True
数据规模： (96372, 12)
订单ID是否唯一： True
标签分布：
is_delayed
0    89839
1     6533
Name: count, dtype: int64


In [40]:
# 1. 将获批时间转换为 datetime 类型
df["order_approved_at"] = pd.to_datetime(
    df["order_approved_at"],
    errors="coerce"
)

# 2. 检查转换后的数据类型
print("时间字段类型：")
print(df["order_approved_at"].dtype)

# 3. 检查是否有无效时间
print("无效时间数量：")
print(df["order_approved_at"].isna().sum())

# 4. 查看时间范围
print("最早获批时间：")
print(df["order_approved_at"].min())

print("最晚获批时间：")
print(df["order_approved_at"].max())

# 5. 按获批时间排序
df = df.sort_values(
    ["order_approved_at", "order_id"]
).reset_index(drop=True)

print("\n排序后的前5条获批时间：")
print(df["order_approved_at"].head())

时间字段类型：
datetime64[us]
无效时间数量：
0
最早获批时间：
2016-09-15 12:16:38
最晚获批时间：
2018-08-29 15:10:26

排序后的前5条获批时间：
0   2016-09-15 12:16:38
1   2016-10-04 09:43:32
2   2016-10-04 10:19:23
3   2016-10-04 10:25:46
4   2016-10-04 10:28:07
Name: order_approved_at, dtype: datetime64[us]


In [41]:
# 总样本数
n = len(df)

# 计算划分位置
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# 确定两个时间分界点
val_start = df.iloc[train_end]["order_approved_at"]
test_start = df.iloc[val_end]["order_approved_at"]

print("验证集开始时间：", val_start)
print("测试集开始时间：", test_start)

验证集开始时间： 2018-04-16 02:15:08
测试集开始时间： 2018-06-21 09:59:08


In [42]:
# 训练集
train = df[
    df["order_approved_at"] < val_start
].copy()

# 验证集
val = df[
    (df["order_approved_at"] >= val_start)
    & (df["order_approved_at"] < test_start)
].copy()

# 测试集
test = df[
    df["order_approved_at"] >= test_start
].copy()

# 检查划分结果
print("训练集：", train.shape)
print("验证集：", val.shape)
print("测试集：", test.shape)

print("\n样本总数是否一致：")
print(len(train) + len(val) + len(test) == len(df))

print("\n三个数据集的延迟率：")
print("Train:", train["is_delayed"].mean())
print("Validation:", val["is_delayed"].mean())
print("Test:", test["is_delayed"].mean())

训练集： (67460, 12)
验证集： (14456, 12)
测试集： (14456, 12)

样本总数是否一致：
True

三个数据集的延迟率：
Train: 0.07835754521197746
Validation: 0.04344216934144992
Test: 0.04281959048146099


In [43]:
# 读取实际送达时间
delivery_info = pd.read_csv(
    project_root / "data/processed/orders_with_delay.csv",
    usecols=["order_id", "order_delivered_customer_date"]
)

# 检查订单ID唯一
assert delivery_info["order_id"].is_unique

# 将实际送达时间关联到建模数据
df = df.merge(
    delivery_info,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# 转换日期类型
df["order_delivered_customer_date"] = pd.to_datetime(
    df["order_delivered_customer_date"],
    errors="coerce"
)

# 检查关联结果
print("关联后数据规模：", df.shape)

print("实际送达时间缺失数量：")
print(df["order_delivered_customer_date"].isna().sum())

关联后数据规模： (96372, 13)
实际送达时间缺失数量：
0


In [44]:
# 训练集：验证期开始前已经送达
train_final = df[
    (df["order_approved_at"] < val_start)
    & (df["order_delivered_customer_date"] < val_start)
].copy()

# 验证集：测试期开始前已经送达
val_final = df[
    (df["order_approved_at"] >= val_start)
    & (df["order_approved_at"] < test_start)
    & (df["order_delivered_customer_date"] < test_start)
].copy()

# 测试集保持原来的时间划分
test_final = df[
    df["order_approved_at"] >= test_start
].copy()

# 统计排除数量
print("训练集排除数量：", len(train) - len(train_final))
print("验证集排除数量：", len(val) - len(val_final))

# 最终样本量
print("\n最终训练集：", train_final.shape)
print("最终验证集：", val_final.shape)
print("最终测试集：", test_final.shape)

# 检查训练集和验证集的延迟比例
print("\n训练集延迟率：", train_final["is_delayed"].mean())
print("验证集延迟率：", val_final["is_delayed"].mean())

训练集排除数量： 3079
验证集排除数量： 1912

最终训练集： (64381, 13)
最终验证集： (12544, 13)
最终测试集： (14456, 13)

训练集延迟率： 0.06815675432192728
验证集延迟率： 0.04296875


In [45]:
# 已经确定的9个特征
model_features = [
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "approval_hour",
    "customer_state",
    "customer_city",
    "item_count",
    "product_amount",
    "freight_amount"
]

# 训练集
X_train = train_final[model_features].copy()
y_train = train_final["is_delayed"].copy()

# 验证集
X_val = val_final[model_features].copy()
y_val = val_final["is_delayed"].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (64381, 9)
y_train: (64381,)
X_val: (12544, 9)
y_val: (12544,)


In [46]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
from sklearn.dummy import DummyClassifier

# 创建模型
dummy = DummyClassifier(strategy="prior")

# 训练模型
dummy.fit(X_train, y_train)

# 输出训练集类别比例
print("模型类别：", dummy.classes_)
print("训练集类别比例：", dummy.class_prior_)

# 预测验证集的类别
y_pred = dummy.predict(X_val)

# 预测验证集的延迟概率
positive_idx = list(dummy.classes_).index(1)
y_prob = dummy.predict_proba(X_val)[:, positive_idx]

print("预测类别前10个：", y_pred[:10])
print("预测延迟概率前10个：", y_prob[:10])

模型类别： [0 1]
训练集类别比例： [0.93184325 0.06815675]
预测类别前10个： [0 0 0 0 0 0 0 0 0 0]
预测延迟概率前10个： [0.06815675 0.06815675 0.06815675 0.06815675 0.06815675 0.06815675
 0.06815675 0.06815675 0.06815675 0.06815675]


In [48]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    average_precision_score,
    confusion_matrix
)

# 1. 准确率
accuracy = accuracy_score(y_val, y_pred)

# 2. 精确率
precision = precision_score(
    y_val, y_pred, zero_division=0
)

# 3. 召回率
recall = recall_score(
    y_val, y_pred, zero_division=0
)

# 4. 平均精确率 AP
ap = average_precision_score(y_val, y_prob)

# 5. 混淆矩阵
cm = confusion_matrix(
    y_val, y_pred, labels=[0, 1]
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("Average Precision:", ap)

print("\nConfusion Matrix:")
print(cm)


Accuracy: 0.95703125
Precision: 0.0
Recall: 0.0
Average Precision: 0.04296875

Confusion Matrix:
[[12005     0]
 [  539     0]]


# 订单延迟预测：基线模型构建与评估

## 1. 基线模型构建

本阶段使用 `scikit-learn` 中的 `DummyClassifier` 构建订单延迟预测的基线模型（Baseline Model），用于为后续机器学习模型提供性能比较基准。

预测任务为二分类问题，目标变量为 `is_delayed`：

- `0`：订单未延迟。
- `1`：订单延迟。

### 1.1 模型选择

采用 `DummyClassifier`，设置：

```python
dummy = DummyClassifier(strategy="prior")
```

`strategy="prior"` 表示模型根据训练集中各类别的比例进行预测，不学习输入特征与目标变量之间的关系。

### 1.2 模型训练

使用训练集特征 `X_train` 和标签 `y_train` 拟合模型：

```python
dummy.fit(X_train, y_train)
```

训练完成后，模型识别出两个类别：

```text
[0 1]
```

训练集类别比例为：

| 类别 | 比例 |
|---|---:|
| 未延迟（0） | 93.1843% |
| 延迟（1） | 6.8157% |

训练集中未延迟订单占绝大多数，因此基线模型将多数类 `0` 作为预测类别。

---

## 2. 模型预测

### 2.1 类别预测

使用：

```python
y_pred = dummy.predict(X_val)
```

模型预测结果均为：

```text
0
```

即所有验证订单均被预测为未延迟。

### 2.2 延迟概率预测

使用：

```python
positive_idx = list(dummy.classes_).index(1)
y_prob = dummy.predict_proba(X_val)[:, positive_idx]
```

模型对每笔订单输出相同的延迟概率：

```text
0.06815675
```

即约为 **6.82%**。

由于 `DummyClassifier(strategy="prior")` 不使用订单特征区分不同样本，因此所有订单得到相同的风险评分。

---

## 3. 模型评估

采用以下指标评价基线模型：

- Accuracy（准确率）
- Precision（精确率）
- Recall（召回率）
- Average Precision（AP，平均精确率）
- Confusion Matrix（混淆矩阵）

### 3.1 评估结果

| 评估指标 | 结果 |
|---|---:|
| 验证集样本数量 | 12,544 |
| Accuracy | 0.95703125 |
| Precision | 0.0000 |
| Recall | 0.0000 |
| Average Precision | 0.04296875 |

### 3.2 混淆矩阵

模型输出：

```text
[[12005     0]
 [  539     0]]
```

其中：

| 实际类别 | 预测未延迟（0） | 预测延迟（1） |
|---|---:|---:|
| 未延迟（0） | 12,005 | 0 |
| 延迟（1） | 539 | 0 |

对应结果：

- TN（True Negative）：12,005
- FP（False Positive）：0
- FN（False Negative）：539
- TP（True Positive）：0

模型正确预测了 12,005 笔未延迟订单，但未能识别出任何真正延迟的订单。

---

## 4. 评估结果分析

### 4.1 Accuracy

模型准确率为：

\[
\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}
\]

代入实际结果：

\[
\text{Accuracy}=\frac{12005}{12544}
=95.703125\%
\]

虽然准确率达到约 **95.70%**，但这是因为验证集中未延迟订单占比较高，模型始终预测多数类也能获得较高准确率。

因此，单独使用 Accuracy 无法充分衡量模型识别延迟订单的能力。

### 4.2 Precision

精确率定义为：

\[
\text{Precision}=\frac{TP}{TP+FP}
\]

由于模型没有预测任何延迟订单：

\[
TP=0,\quad FP=0
\]

此时精确率在数学上未定义。评估代码设置了 `zero_division=0`，因此返回：

\[
\text{Precision}=0
\]

### 4.3 Recall

召回率定义为：

\[
\text{Recall}=\frac{TP}{TP+FN}
\]

代入实际结果：

\[
\text{Recall}=\frac{0}{0+539}=0
\]

说明验证集中 539 笔真正延迟的订单均未被模型识别。

### 4.4 Average Precision

模型的 AP 为：

\[
\text{AP}=0.04296875
\]

即约 **4.30%**。

由于模型对全部验证订单输出相同的延迟风险评分，无法有效区分高风险和低风险订单。

在这种常数评分情况下，AP 等于验证集正样本比例：

\[
\text{AP}=\frac{539}{12544}
=0.04296875
\]

因此，当前 AP 可作为后续模型风险排序能力的基准。

---

## 5. 总结

本阶段成功构建并评估了 `DummyClassifier` 基线模型。

主要结论如下：

1. 训练集中未延迟订单占比约为 93.18%，存在明显的类别不平衡。
2. 基线模型始终预测订单未延迟，验证集 Accuracy 达到约 95.70%。
3. 模型未能识别任何延迟订单，Recall 为 0。
4. 模型对所有订单输出相同的延迟风险评分，AP 约为 4.30%。
5. 在类别不平衡的订单延迟预测任务中，仅使用 Accuracy 不足以衡量模型的实际识别能力。

**最终结论：**

`DummyClassifier` 虽然具有较高的准确率，但没有学习订单特征与配送延迟之间的关系，无法有效识别延迟订单。

本阶段得到的评估结果将作为后续逻辑回归模型的比较基准，重点比较 AP、Precision、Recall 和混淆矩阵等指标。

In [49]:
numeric_features = [
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "approval_hour",
    "item_count",
    "product_amount",
    "freight_amount"
]

categorical_features = [
    "customer_state",
    "customer_city"
]

model_features = numeric_features + categorical_features

X_train = train_final[model_features].copy()
y_train = train_final["is_delayed"].copy()

X_val = val_final[model_features].copy()
y_val = val_final["is_delayed"].copy()

print("训练集特征：", X_train.shape)
print("验证集特征：", X_val.shape)

print("\n数值特征类型：")
print(X_train[numeric_features].dtypes)

print("\n类别特征类型：")
print(X_train[categorical_features].dtypes)

print("\n训练集缺失情况：")
print(X_train.isna().sum())

训练集特征： (64381, 9)
验证集特征： (12544, 9)

数值特征类型：
purchase_month        int64
purchase_weekday      int64
purchase_hour         int64
approval_hour         int64
item_count          float64
product_amount      float64
freight_amount      float64
dtype: object

类别特征类型：
customer_state    str
customer_city     str
dtype: object

训练集缺失情况：
purchase_month      0
purchase_weekday    0
purchase_hour       0
approval_hour       0
item_count          0
product_amount      0
freight_amount      0
customer_state      0
customer_city       0
dtype: int64


In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy import sparse

# 1. 数值特征：标准化
numeric_transformer = StandardScaler()

# 2. 类别特征：One-Hot 编码
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

# 3. 组合两种预处理
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    sparse_threshold=1.0
)

# 4. 只用训练集学习预处理参数
X_train_processed = preprocessor.fit_transform(X_train)

# 5. 验证集使用训练集学到的规则
X_val_processed = preprocessor.transform(X_val)

# 6. 检查结果
print("处理前训练集：", X_train.shape)
print("处理后训练集：", X_train_processed.shape)

print("处理前验证集：", X_val.shape)
print("处理后验证集：", X_val_processed.shape)

print("训练集是否为稀疏矩阵：",
      sparse.issparse(X_train_processed))


处理前训练集： (64381, 9)
处理后训练集： (64381, 3710)
处理前验证集： (12544, 9)
处理后验证集： (12544, 3710)
训练集是否为稀疏矩阵： True


In [51]:
from sklearn.linear_model import LogisticRegression

# 1. 创建逻辑回归模型
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# 2. 使用训练集训练模型
lr_model.fit(X_train_processed, y_train)

# 3. 查看模型训练情况
print("模型训练完成！")
print("实际迭代次数：", lr_model.n_iter_)
print("模型系数形状：", lr_model.coef_.shape)
print("模型截距：", lr_model.intercept_)


模型训练完成！
实际迭代次数： [122]
模型系数形状： (1, 3710)
模型截距： [-2.62673329]


In [52]:
# 1. 预测验证集的类别
lr_pred = lr_model.predict(X_val_processed)

# 2. 预测验证集的延迟概率
positive_idx = list(lr_model.classes_).index(1)

lr_prob = lr_model.predict_proba(
    X_val_processed
)[:, positive_idx]

# 3. 查看预测结果
print("预测类别前10个：")
print(lr_pred[:10])

print("\n预测延迟概率前10个：")
print(lr_prob[:10])

# 4. 统计预测为延迟的订单数量
print("\n预测为延迟的订单数：")
print((lr_pred == 1).sum())

print("\n实际延迟的订单数：")
print((y_val == 1).sum())


预测类别前10个：
[0 0 0 0 0 0 0 0 0 0]

预测延迟概率前10个：
[0.04730104 0.12351862 0.04675481 0.11904609 0.08463316 0.04740868
 0.10338678 0.03586544 0.14514639 0.05451917]

预测为延迟的订单数：
0

实际延迟的订单数：
539


In [53]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    average_precision_score,
    confusion_matrix
)

# 1. 计算评价指标
lr_accuracy = accuracy_score(y_val, lr_pred)

lr_precision = precision_score(
    y_val, lr_pred, zero_division=0
)

lr_recall = recall_score(
    y_val, lr_pred, zero_division=0
)

lr_ap = average_precision_score(
    y_val, lr_prob
)

lr_cm = confusion_matrix(
    y_val, lr_pred, labels=[0, 1]
)

# 2. 输出结果
print("Logistic Regression 评估结果：")

print("Accuracy:", lr_accuracy)
print("Precision:", lr_precision)
print("Recall:", lr_recall)
print("Average Precision:", lr_ap)

print("\nConfusion Matrix:")
print(lr_cm)

# 3. 检查预测概率范围
print("\n最小预测概率：", lr_prob.min())
print("最大预测概率：", lr_prob.max())


Logistic Regression 评估结果：
Accuracy: 0.95703125
Precision: 0.0
Recall: 0.0
Average Precision: 0.06864518018140023

Confusion Matrix:
[[12005     0]
 [  539     0]]

最小预测概率： 0.008950777675826676
最大预测概率： 0.4892276523551179


In [54]:
#生成正式的模型对比表
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    average_precision_score
)

comparison = pd.DataFrame({
    "Model": [
        "DummyClassifier",
        "LogisticRegression"
    ],
    "Accuracy": [
        accuracy_score(y_val, y_pred),
        accuracy_score(y_val, lr_pred)
    ],
    "Precision": [
        precision_score(y_val, y_pred, zero_division=0),
        precision_score(y_val, lr_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_val, y_pred, zero_division=0),
        recall_score(y_val, lr_pred, zero_division=0)
    ],
    "AP": [
        average_precision_score(y_val, y_prob),
        average_precision_score(y_val, lr_prob)
    ]
})

display(comparison)

,Model,Accuracy,Precision,Recall,AP
0,DummyClassifier,0.957031,0.0,0.0,0.042969
1,LogisticRegression,0.957031,0.0,0.0,0.068645


# 逻辑回归模型训练与评估

## 1. 实验目的

在前期构建的 `DummyClassifier` 基线模型基础上，使用逻辑回归（Logistic Regression）进行订单延迟预测。

本阶段主要目标：

1. 对数值特征和类别特征进行预处理。
2. 训练逻辑回归二分类模型。
3. 使用验证集评估模型的预测表现。
4. 与 `DummyClassifier` 基线模型比较，判断逻辑回归是否能够提升延迟风险识别能力。

---

## 2. 数据集与特征

### 2.1 数据规模

本阶段使用经过异常订单筛选和标签可用性检查后的训练集与验证集。

| 数据集 | 样本数量 | 原始特征数量 |
|---|---:|---:|
| 训练集 | 64,381 | 9 |
| 验证集 | 12,544 | 9 |

目标变量为 `is_delayed`：

- `0`：订单未延迟。
- `1`：订单延迟。

### 2.2 模型输入特征

共使用 9 个特征。

**数值特征（7 个）：**

- `purchase_month`
- `purchase_weekday`
- `purchase_hour`
- `approval_hour`
- `item_count`
- `product_amount`
- `freight_amount`

**类别特征（2 个）：**

- `customer_state`
- `customer_city`

训练集中的上述特征均不存在缺失值。

---

## 3. 数据预处理

由于逻辑回归要求输入数值型特征，因此采用 `ColumnTransformer` 对不同类型特征进行处理。

### 3.1 数值特征标准化

使用 `StandardScaler` 对 7 个数值特征进行标准化：

\[
z=\frac{x-\mu}{\sigma}
\]

其中，\(\mu\) 和 \(\sigma\) 分别为训练集对应特征的均值和标准差。

### 3.2 类别特征编码

使用 `OneHotEncoder` 对客户所在州和城市进行独热编码。

设置：

```python
OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)
```

该方法将类别变量转换为逻辑回归可以处理的数值特征。

### 3.3 预处理结果

| 数据集 | 处理前 | 处理后 |
|---|---|---|
| 训练集 | (64381, 9) | (64381, 3710) |
| 验证集 | (12544, 9) | (12544, 3710) |

其中：

- 7 列为标准化后的数值特征。
- 3,703 列为类别编码特征。
- 最终特征矩阵采用稀疏矩阵存储。

预处理器仅在训练集上使用 `fit_transform()`，验证集只使用 `transform()`，避免验证集信息参与预处理参数的拟合。

---

## 4. 逻辑回归模型训练

使用 `scikit-learn` 的 `LogisticRegression`：

```python
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_processed, y_train)
```

### 4.1 模型训练结果

| 项目 | 结果 |
|---|---|
| 最大迭代次数 | 1,000 |
| 实际迭代次数 | 122 |
| 模型系数形状 | (1, 3710) |
| 模型截距 | -2.62673329 |

模型成功完成训练，未达到设置的最大迭代次数。

### 4.2 预测结果

逻辑回归为不同订单输出了不同的延迟预测概率。

验证集预测概率范围：

| 指标 | 结果 |
|---|---:|
| 最小预测概率 | 0.008951 |
| 最大预测概率 | 0.489228 |

采用默认分类阈值 0.5 时：

- 预测为延迟的订单：0 笔。
- 实际延迟的订单：539 笔。

由于所有预测概率均低于 0.5，模型将全部验证订单判定为未延迟。

---

## 5. 模型评估结果

采用 Accuracy、Precision、Recall、Average Precision（AP）和混淆矩阵进行评估。

### 5.1 逻辑回归评估指标

| 指标 | 结果 |
|---|---:|
| Accuracy | 0.957031 |
| Precision | 0.000000 |
| Recall | 0.000000 |
| Average Precision | 0.068645 |

### 5.2 混淆矩阵

```text
[[12005     0]
 [  539     0]]
```

具体含义：

| 指标 | 数量 |
|---|---:|
| TN：正确预测未延迟 | 12,005 |
| FP：误报延迟 | 0 |
| FN：漏报延迟 | 539 |
| TP：正确预测延迟 | 0 |

在默认阈值下，逻辑回归没有成功识别延迟订单，Recall 为 0。

但这并不意味着模型没有学习到有效信息，因为 AP 衡量的是不同订单预测风险评分的排序表现，而不依赖固定的 0.5 分类阈值。

---

## 6. 与基线模型对比

### 6.1 模型性能对比

| Model | Accuracy | Precision | Recall | AP |
|---|---:|---:|---:|---:|
| DummyClassifier | 0.957031 | 0.000000 | 0.000000 | 0.042969 |
| LogisticRegression | 0.957031 | 0.000000 | 0.000000 | **0.068645** |

### 6.2 对比分析

**（1）Accuracy 没有变化**

两个模型的准确率均约为 95.70%。

原因是两个模型在默认分类规则下，均将全部验证订单预测为未延迟。

由于验证集中未延迟订单占绝大多数，即使模型无法识别延迟订单，也能获得较高的准确率。

**（2）Precision 和 Recall 均为 0**

两个模型均未预测出延迟订单。

因此，虽然整体准确率较高，但尚未实现对延迟订单的有效识别。

**（3）逻辑回归的 AP 有所提高**

基线模型：

\[
AP_{\text{Dummy}}=0.042969
\]

逻辑回归：

\[
AP_{\text{LR}}=0.068645
\]

AP 绝对提升约为：

\[
0.068645-0.042969=0.025676
\]

即 **2.57 个百分点**。

相对提升约为：

\[
\frac{0.068645-0.042969}{0.042969}
\times100\%
\approx59.8\%
\]

说明逻辑回归相比基线模型具有一定的延迟风险排序能力。

不过，AP 的绝对值仍然较低，当前结果不足以证明模型具有良好的实际业务预测效果。

---

## 7. 现阶段结论

本阶段成功完成了逻辑回归模型训练和验证集评估。

主要结论如下：

1. 成功将 9 个原始特征转换为 3,710 个数值特征，并采用稀疏矩阵存储。
2. 逻辑回归模型经过 122 次迭代完成训练。
3. 模型能够为不同订单输出不同的延迟风险评分。
4. 默认 0.5 分类阈值下，模型未识别出任何延迟订单。
5. 逻辑回归的 AP 从基线模型的 4.30% 提升至约 6.86%。
6. 虽然风险排序能力有所提高，但分类识别效果仍有改善空间。

**总体结论：**

逻辑回归相比 `DummyClassifier` 具有一定的延迟风险区分能力，但默认阈值下无法识别延迟订单。后续需要进一步研究分类阈值对 Precision、Recall、误报和漏报数量的影响。

## 8. 下一阶段工作

下一阶段将基于验证集预测概率开展：

1. 不同分类阈值下的模型性能比较。
2. Precision 与 Recall 的权衡分析。
3. 误报和漏报订单的错误分析。
4. 根据验证集结果选择合理的分类阈值。

测试集将保留用于最终模型评估，不在当前阶段用于模型选择。

In [55]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix
)

# 设置需要比较的分类阈值
thresholds = [0.05, 0.08, 0.10, 0.15, 0.20, 0.50]

results = []

for threshold in thresholds:

    # 根据当前阈值生成预测标签
    y_pred_threshold = (lr_prob >= threshold).astype(int)

    # 计算 Precision
    precision = precision_score(
        y_val, y_pred_threshold, zero_division=0
    )

    # 计算 Recall
    recall = recall_score(
        y_val, y_pred_threshold, zero_division=0
    )

    # 计算混淆矩阵
    tn, fp, fn, tp = confusion_matrix(
        y_val, y_pred_threshold, labels=[0, 1]
    ).ravel()

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "predicted_delayed": int(y_pred_threshold.sum()),
        "TP": tp,
        "FP": fp,
        "FN": fn
    })

# 整理为 DataFrame
threshold_results = pd.DataFrame(results)

display(threshold_results)


,threshold,precision,recall,predicted_delayed,TP,FP,FN
0,0.05,0.059736,0.679035,6127,366,5761,173
1,0.08,0.065000,0.410019,3400,221,3179,318
2,0.10,0.060062,0.285714,2564,154,2410,385
3,0.15,0.096509,0.087199,487,47,440,492
4,0.20,0.109290,0.037106,183,20,163,519
5,0.50,0.000000,0.000000,0,0,0,539


In [56]:
# 临时选择分析阈值
threshold = 0.15

# 创建验证集分析表
val_analysis = val_final[
    [
        "order_id",
        "customer_state",
        "customer_city",
        "item_count",
        "product_amount",
        "freight_amount",
        "is_delayed"
    ]
].copy()

# 加入模型预测概率和预测类别
val_analysis["predicted_probability"] = lr_prob

val_analysis["predicted_label"] = (
    lr_prob >= threshold
).astype(int)

# 误报：实际不延迟，但预测延迟
fp_orders = val_analysis[
    (val_analysis["is_delayed"] == 0)
    & (val_analysis["predicted_label"] == 1)
]

# 漏报：实际延迟，但预测不延迟
fn_orders = val_analysis[
    (val_analysis["is_delayed"] == 1)
    & (val_analysis["predicted_label"] == 0)
]

print("误报数量：", len(fp_orders))
print("漏报数量：", len(fn_orders))

print("\n误报样本（概率最高的5笔）：")
display(
    fp_orders.sort_values(
        "predicted_probability",
        ascending=False
    ).head(5)
)

print("\n漏报样本（概率最低的5笔）：")
display(
    fn_orders.sort_values(
        "predicted_probability"
    ).head(5)
)


误报数量： 440
漏报数量： 492

误报样本（概率最高的5笔）：


,order_id,customer_state,customer_city,item_count,product_amount,freight_amount,is_delayed,predicted_probability,predicted_label
74513,43887f3ff49629a7d608220b3a832e5f,RJ,armacao dos buzios,1.0,325.0,45.85,0,0.489228,1
78067,71a88ca55a16c100b22fa5dace71266a,RJ,armacao dos buzios,1.0,572.9,21.89,0,0.454942,1
74691,835dcecf534791041554b984434d72a0,RJ,armacao dos buzios,1.0,94.9,15.36,0,0.435409,1
68182,2e0f7f4c596ba8da4bd2be65336fec1b,AL,maceio,1.0,189.9,133.84,0,0.387995,1
68953,1efe916e0147732877ae7c2ae6a245ba,AL,maceio,1.0,750.0,106.89,0,0.378336,1



漏报样本（概率最低的5笔）：


,order_id,customer_state,customer_city,item_count,product_amount,freight_amount,is_delayed,predicted_probability,predicted_label
70375,8adc2cd3ee5e7b81b1d476e6878714fa,SP,franco da rocha,1.0,39.90,7.87,1,0.012237,0
70306,afc2b80ab67a5a0cf0d36422cd6409eb,SP,taboao da serra,1.0,18.95,13.47,1,0.015370,0
69314,666e488ce51569ca13755283bce0e7f0,SP,sorocaba,1.0,26.90,8.88,1,0.016530,0
75464,c49be9a11fd13933307cc6a19b03a895,MG,vicosa,6.0,834.00,94.68,1,0.017615,0
76112,fb7688c98b85305cf636fb26b2804186,MG,alfenas,1.0,30.00,18.23,1,0.019303,0


In [57]:
import numpy as np

# 为验证集每笔订单标记预测结果类型
val_analysis["error_type"] = np.select(
    [
        (val_analysis["is_delayed"] == 0)
        & (val_analysis["predicted_label"] == 1),

        (val_analysis["is_delayed"] == 1)
        & (val_analysis["predicted_label"] == 0),

        (val_analysis["is_delayed"] == 1)
        & (val_analysis["predicted_label"] == 1)
    ],
    ["FP", "FN", "TP"],
    default="TN"
)

# 按预测结果类型进行统计
error_summary = (
    val_analysis
    .groupby("error_type")
    .agg(
        order_count=("order_id", "count"),
        avg_product_amount=("product_amount", "mean"),
        median_product_amount=("product_amount", "median"),
        avg_freight=("freight_amount", "mean"),
        median_freight=("freight_amount", "median"),
        avg_item_count=("item_count", "mean"),
        avg_pred_prob=("predicted_probability", "mean")
    )
    .round(3)
)

display(error_summary)

,order_count,avg_product_amount,median_product_amount,avg_freight,median_freight,avg_item_count,avg_pred_prob
error_type,,,,,,,
FN,492,141.596,99.0,23.908,18.695,1.106,0.072
FP,440,296.880,123.9,46.490,24.515,1.136,0.197
TN,11565,137.488,89.9,21.579,17.930,1.165,0.060
TP,47,180.075,99.9,28.575,23.280,1.021,0.206


In [58]:
# 计算 F1-score
p = threshold_results["precision"]
r = threshold_results["recall"]

threshold_results["f1"] = (
    2 * p * r / (p + r)
).fillna(0)

# 计算报警订单占验证集的比例
threshold_results["alert_rate"] = (
    threshold_results["predicted_delayed"] / len(y_val)
)

# 查看比较结果
display(
    threshold_results[
        [
            "threshold",
            "precision",
            "recall",
            "f1",
            "alert_rate",
            "TP",
            "FP",
            "FN"
        ]
    ].round(4)
)

# 找到当前候选阈值中 F1 最高的阈值
best_idx = threshold_results["f1"].idxmax()

print("当前候选中 F1 最高的阈值：")
print(threshold_results.loc[best_idx])

,threshold,precision,recall,f1,alert_rate,TP,FP,FN
0,0.05,0.0597,0.6790,0.1098,0.4884,366,5761,173
1,0.08,0.0650,0.4100,0.1122,0.2710,221,3179,318
2,0.10,0.0601,0.2857,0.0993,0.2044,154,2410,385
3,0.15,0.0965,0.0872,0.0916,0.0388,47,440,492
4,0.20,0.1093,0.0371,0.0554,0.0146,20,163,519
5,0.50,0.0000,0.0000,0.0000,0.0000,0,0,539


当前候选中 F1 最高的阈值：
threshold               0.080000
precision               0.065000
recall                  0.410019
predicted_delayed    3400.000000
TP                    221.000000
FP                   3179.000000
FN                    318.000000
f1                      0.112211
alert_rate              0.271046
Name: 1, dtype: float64


In [59]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 根据验证集 F1 结果暂定阈值
selected_threshold = 0.08

# 生成预测标签
val_pred_selected = (
    lr_prob >= selected_threshold
).astype(int)

# 计算指标
print("选定阈值：", selected_threshold)

print("Precision:",
      precision_score(y_val, val_pred_selected))

print("Recall:",
      recall_score(y_val, val_pred_selected))

print("F1:",
      f1_score(y_val, val_pred_selected))

print("混淆矩阵：")
print(confusion_matrix(
    y_val,
    val_pred_selected,
    labels=[0, 1]
))

选定阈值： 0.08
Precision: 0.065
Recall: 0.4100185528756957
F1: 0.11221122112211221
混淆矩阵：
[[8826 3179]
 [ 318  221]]


# 分类阈值优化与预测错误分析

## 1. 实验目的

在第 11 天逻辑回归模型训练与评估的基础上，进一步分析模型在不同分类阈值下的预测表现，并研究模型产生误报和漏报的情况。

本阶段主要完成：

1. 比较不同分类阈值下的 Precision、Recall 和 F1-score。
2. 分析分类阈值对误报、漏报以及报警订单数量的影响。
3. 提取典型预测错误样本。
4. 对 FP、FN、TP、TN 四类订单进行特征统计。
5. 在验证集上确定第一版模型的暂定分类阈值。

本阶段仅使用验证集进行分析，不使用测试集进行阈值选择。

---

## 2. 数据与模型说明

### 2.1 数据规模

本阶段使用已经训练完成的逻辑回归模型，以及对应的验证集预测概率 `lr_prob`。

| 指标 | 数量 |
|---|---:|
| 训练集订单数 | 64,381 |
| 验证集订单数 | 12,544 |
| 验证集未延迟订单 | 12,005 |
| 验证集延迟订单 | 539 |
| 验证集延迟率 | 4.30% |

### 2.2 模型基本表现

上一阶段逻辑回归模型的评估结果为：

| 指标 | 结果 |
|---|---:|
| Accuracy | 0.957031 |
| Precision | 0 |
| Recall | 0 |
| Average Precision（AP） | 0.068645 |

逻辑回归输出的最大延迟预测概率约为 `0.489228`。

由于默认分类阈值为 0.5，所有验证订单均被预测为未延迟，导致 Recall 为 0。

因此，本阶段通过调整分类阈值，研究模型对延迟订单的识别能力。

---

## 3. 分类阈值分析

### 3.1 分类阈值定义

设模型预测订单延迟的概率为 \(p\)，分类阈值为 \(t\)，则：

\[
\hat y =
\begin{cases}
1, & p\geq t\\
0, & p<t
\end{cases}
\]

其中：

- `1` 表示预测订单延迟。
- `0` 表示预测订单未延迟。

降低阈值通常会增加预测为延迟的订单数量，提高 Recall，但也可能增加误报。

### 3.2 阈值实验设置

选取以下六个候选阈值：

```python
thresholds = [0.05, 0.08, 0.10, 0.15, 0.20, 0.50]
```

针对每个阈值，计算 Precision、Recall、F1-score、TP、FP、FN 和报警比例。

### 3.3 阈值比较结果

| 阈值 | Precision | Recall | F1 | 报警比例 | TP | FP | FN |
|---|---:|---:|---:|---:|---:|---:|---:|
| 0.05 | 5.97% | 67.90% | 10.98% | 48.84% | 366 | 5,761 | 173 |
| **0.08** | **6.50%** | **41.00%** | **11.22%** | **27.10%** | **221** | **3,179** | **318** |
| 0.10 | 6.01% | 28.57% | 9.93% | 20.44% | 154 | 2,410 | 385 |
| 0.15 | 9.65% | 8.72% | 9.16% | 3.88% | 47 | 440 | 492 |
| 0.20 | 10.93% | 3.71% | 5.54% | 1.46% | 20 | 163 | 519 |
| 0.50 | 0% | 0% | 0% | 0% | 0 | 0 | 539 |

### 3.4 实验结果分析

**（1）低阈值可以提高召回率**

当阈值为 0.05 时：

- 成功识别 366 笔延迟订单。
- Recall 达到 67.90%。
- 误报数量达到 5,761 笔。
- 约 48.84% 的验证订单被标记为延迟。

该阈值虽然能够识别较多延迟订单，但报警规模较大。

**（2）高阈值能够减少报警，但召回率降低**

当阈值提高到 0.20 时：

- 仅有 183 笔订单被预测为延迟。
- 其中 20 笔真正延迟。
- Precision 为 10.93%。
- Recall 降至 3.71%。

说明该阈值只能识别很少一部分延迟订单。

**（3）Precision 并不随阈值单调变化**

实验中，阈值 0.08 的 Precision 为 6.50%，高于阈值 0.10 的 6.01%。

这是因为改变阈值时，被移出报警集合的订单中，真实延迟与未延迟的比例不一定保持一致。

因此，阈值选择应综合考虑多个指标，而不能只看阈值大小。

---

## 4. 预测错误样本分析

为了进一步研究模型的误报和漏报情况，首先使用阈值 `0.15` 提取典型错误订单。

选择该阈值仅用于错误案例分析，不代表最终分类阈值。

### 4.1 错误类型定义

| 类型 | 含义 |
|---|---|
| TP | 实际延迟，预测延迟 |
| FP | 实际未延迟，预测延迟 |
| TN | 实际未延迟，预测未延迟 |
| FN | 实际延迟，预测未延迟 |

### 4.2 阈值 0.15 下的预测结果

| 类型 | 数量 |
|---|---:|
| TP | 47 |
| FP | 440 |
| TN | 11,565 |
| FN | 492 |

### 4.3 典型误报样本

选取预测延迟概率最高的 5 笔误报订单：

| 客户州 | 商品金额 | 预测延迟概率 | 实际标签 |
|---|---:|---:|---:|
| RJ | 325.00 | 48.92% | 0 |
| RJ | 572.90 | 45.49% | 0 |
| RJ | 94.90 | 43.54% | 0 |
| AL | 189.90 | 38.80% | 0 |
| AL | 750.00 | 37.83% | 0 |

这些订单实际没有延迟，但模型预测风险较高。

样本中出现了部分金额较高的订单，并集中于 RJ、AL 两个州。

但由于仅观察了五笔高概率误报样本，无法据此判断整体误报是否集中在这些地区，也不能认定金额直接造成了误报。

### 4.4 典型漏报样本

选取预测延迟概率最低的 5 笔漏报订单：

| 客户州 | 商品金额 | 预测延迟概率 | 实际标签 |
|---|---:|---:|---:|
| SP | 39.90 | 1.22% | 1 |
| SP | 18.95 | 1.54% | 1 |
| SP | 26.90 | 1.65% | 1 |
| MG | 834.00 | 1.76% | 1 |
| MG | 30.00 | 1.93% | 1 |

这些订单最终发生延迟，但模型预测概率较低。

说明模型对部分真实延迟订单的风险估计明显偏低。

由于上述样本存在金额和地区差异，仅凭少数案例无法确定漏报的具体原因。

---

## 5. 全部错误订单的特征统计

为了避免仅根据个别案例下结论，对验证集中所有订单按照 FP、FN、TP、TN 进行分组统计。

### 5.1 分组统计结果

以下结果均使用阈值 `0.15`。

| 类型 | 订单数 | 平均商品金额 | 商品金额中位数 | 平均运费 | 平均商品数 | 平均预测概率 |
|---|---:|---:|---:|---:|---:|---:|
| FN | 492 | 141.596 | 99.0 | 23.908 | 1.106 | 0.072 |
| FP | 440 | 296.880 | 123.9 | 46.490 | 1.136 | 0.197 |
| TN | 11,565 | 137.488 | 89.9 | 21.579 | 1.165 | 0.060 |
| TP | 47 | 180.075 | 99.9 | 28.575 | 1.021 | 0.206 |

### 5.2 误报订单分析：FP 与 TN 比较

FP 和 TN 均为实际未延迟订单。

统计发现：

- FP 平均商品金额为 296.880，高于 TN 的 137.488。
- FP 平均运费为 46.490，高于 TN 的 21.579。
- FP 商品金额中位数为 123.9，高于 TN 的 89.9。

说明在当前模型和阈值下，误报订单的商品金额和运费整体高于正确识别的未延迟订单。

这一现象表明模型对部分高金额或高运费订单给出了较高的风险评分，但不能仅根据分组均值确定具体原因。

### 5.3 漏报订单分析：FN 与 TP 比较

FN 和 TP 均为实际延迟订单。

统计发现：

- FN 平均商品金额为 141.596，低于 TP 的 180.075。
- FN 平均运费为 23.908，低于 TP 的 28.575。
- FN 平均预测概率为 7.2%，TP 为 20.6%。

这说明在实际延迟订单中，被成功识别的订单和被漏报的订单在部分特征分布上存在差异。

但目前尚不能确定这些差异是否由地区、金额、运费或其他未纳入模型的因素造成。

### 5.4 错误分析小结

本次错误分析发现：

1. 模型存在较多误报和漏报。
2. 误报订单的平均金额和运费高于正确预测的未延迟订单。
3. 部分真实延迟订单的预测概率非常低。
4. 当前特征能够提供一定风险区分信息，但还不足以有效区分所有延迟订单。

上述结果属于统计观察，不代表已确认模型错误的因果原因。

---

## 6. 基于 F1-score 的阈值选择

### 6.1 F1-score 定义

F1-score 是 Precision 和 Recall 的调和平均数：

\[
F_1=\frac{2PR}{P+R}
\]

其中：

- \(P\)：Precision。
- \(R\)：Recall。

本阶段在没有明确业务误报成本和人工处理能力约束的情况下，暂时使用验证集 F1-score 作为第一版阈值选择依据。

### 6.2 候选阈值选择结果

在六个候选阈值中：

\[
\boxed{t=0.08}
\]

对应的 F1-score 最高：

\[
F_1=0.112211
\]

即约为 **11.22%**。

该结果仅表示在当前六个候选阈值中表现最好，不保证是全部可能阈值中的全局最优解。

### 6.3 选定阈值下的模型表现

使用：

```python
selected_threshold = 0.08

val_pred_selected = (
    lr_prob >= selected_threshold
).astype(int)
```

得到：

| 指标 | 结果 |
|---|---:|
| 分类阈值 | 0.08 |
| Precision | 0.065000 |
| Recall | 0.410019 |
| F1-score | 0.112211 |
| 预测延迟订单数量 | 3,400 |
| 真实延迟订单数量 | 539 |
| 成功识别延迟订单 | 221 |
| 误报订单 | 3,179 |
| 漏报订单 | 318 |
| 报警比例 | 27.10% |

### 6.4 最终混淆矩阵

```text
[[8826 3179]
 [ 318  221]]
```

对应：

| 实际类别 | 预测未延迟 | 预测延迟 |
|---|---:|---:|
| 未延迟 | 8,826 | 3,179 |
| 延迟 | 318 | 221 |

模型成功识别了约 41% 的实际延迟订单。

但是，3,400 笔报警订单中只有 221 笔真正延迟，Precision 仅为 6.50%。

因此，当前模型虽然具备一定的延迟风险识别能力，但仍然存在较大的误报成本与漏报问题。

---

## 7. 模型效果与局限分析

### 7.1 模型的有效性

与 `DummyClassifier` 相比，逻辑回归的 AP 从约 4.30% 提高到 6.86%。

说明逻辑回归能够利用订单特征形成一定的风险排序能力。

通过降低分类阈值，模型能够识别一部分真正延迟的订单，而不再全部预测为未延迟。

### 7.2 当前模型的主要局限

**（1）Precision 较低**

在暂定阈值 0.08 下，Precision 只有 6.50%，说明绝大部分报警订单实际上没有发生延迟。

**（2）Recall 仍然有限**

模型仅识别 539 笔实际延迟订单中的 221 笔，仍有 318 笔延迟订单被漏报。

**（3）模型特征有限**

当前模型主要使用订单时间、客户地区、商品数量、金额和运费等特征。

这些特征未必能够充分反映订单履约过程中的全部风险因素。

**（4）缺少明确业务成本约束**

当前采用 F1-score 选择阈值，没有结合真实的人工处理能力、误报成本或漏报损失。

因此，当前选定的阈值并不能直接认定为业务最优阈值。

**（5）验证集性能不等于最终泛化性能**

当前阈值是在验证集上选择的，最终性能仍需要通过独立测试集进行评估。

---

## 8. 本阶段总结

本阶段完成了逻辑回归模型的分类阈值分析与错误样本研究。

主要成果：

1. 比较了六个候选分类阈值。
2. 计算了各阈值的 Precision、Recall、F1-score 及误报、漏报数量。
3. 提取并检查了 5 个误报和 5 个漏报案例。
4. 对全部 FP、FN、TP、TN 订单进行了特征分组统计。
5. 在当前候选阈值中，选择 `0.08` 作为 F1-score 最高的暂定阈值。
6. 在该阈值下，模型 Precision 为 6.50%，Recall 为 41.00%，F1-score 为 11.22%。

**总体结论：**

逻辑回归模型相较于基线模型具有一定的延迟风险排序能力，通过阈值调整能够识别部分延迟订单。

但当前模型的 Precision 和 F1-score 仍然较低，误报和漏报问题明显，尚不足以证明模型具有实际部署价值。

本阶段完成了初步模型分析与阈值选择，为后续独立测试集评估提供基础。

## 9. 下一阶段工作

下一阶段将：

1. 固定当前模型配置与暂定阈值。
2. 使用此前保留的独立测试集进行最终评估。
3. 比较验证集与测试集模型表现。
4. 整理模型训练、预处理及评估脚本。
5. 完善实验报告与项目可复现说明。

测试集不参与本阶段的阈值选择或模型参数调整。